Practical 5 : Perform Classification using ML Pipeline with PySpark in Databricks

In [36]:
#!pip install pyspark

In [37]:
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator


In [38]:
spark = SparkSession.builder.appName("ML Pipeline Classification").getOrCreate()

In [39]:
data = [
    (25, 50000, "IT", 1),
    (30, 60000, "HR", 0),
    (35, 65000, "Finance", 1),
    (40, 70000, "IT", 0),
    (28, 52000, "HR", 1)
]
columns = ["age", "salary", "department", "label"]
df = spark.createDataFrame(data, columns)
df.show()


+---+------+----------+-----+
|age|salary|department|label|
+---+------+----------+-----+
| 25| 50000|        IT|    1|
| 30| 60000|        HR|    0|
| 35| 65000|   Finance|    1|
| 40| 70000|        IT|    0|
| 28| 52000|        HR|    1|
+---+------+----------+-----+



In [40]:
# Convert categorical column to numeric
dept_indexer = StringIndexer(inputCol="department", outputCol="dept_index")

# Combine features
assembler = VectorAssembler(
    inputCols=["age", "salary", "dept_index"],
    outputCol="features"
)

# Scale features
scaler = StandardScaler(inputCol="features", outputCol="scaled_features")


In [41]:
lr = LogisticRegression(featuresCol="scaled_features", labelCol="label")

In [42]:
pipeline = Pipeline(stages=[dept_indexer, assembler, scaler, lr])

In [43]:
train_data, test_data = df.randomSplit([0.8, 0.2])

In [44]:
model = pipeline.fit(train_data)

In [45]:
predictions = model.transform(test_data)

In [46]:
predictions.select("age", "salary", "department", "label", "prediction").show()

+---+------+----------+-----+----------+
|age|salary|department|label|prediction|
+---+------+----------+-----+----------+
| 28| 52000|        HR|    1|       1.0|
+---+------+----------+-----+----------+



In [47]:
evaluator = BinaryClassificationEvaluator(labelCol="label")


accuracy = evaluator.evaluate(predictions)
print("Model Accuracy:", accuracy)


Model Accuracy: 1.0
